In [ ]:
from dotenv import load_dotenv
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings
from langchain_unstructured import UnstructuredLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

load_dotenv()

# 1. load and transform documents
splitter = CharacterTextSplitter.from_tiktoken_encoder(
  separator="\n",
  chunk_size=600, # 600자 청크로 분할
  chunk_overlap=100, # 청크 간 100자 겹침으로 문맥 유지
)

loader = UnstructuredLoader("./files/How_Neflix_Uses_Java_-_2026_.pdf")

docs = filter_complex_metadata(loader.load_and_split(text_splitter=splitter))


# 2. embeddings with caching
cache_dir = LocalFileStore("./.cache/")

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)


# 3. store in vector database
vectorstore = Chroma.from_documents(docs, cached_embeddings)

# 임베딩 작업을 할 때 먼저 캐시에 embeddings가 있는지 확인하고, 없으면 OpenAI API를 호출하여 임베딩을 생성한 후 캐시에 저장한다. 
# 이렇게 하면 동일한 텍스트에 대해 여러 번 임베딩을 생성할 때 API 호출을 줄일 수 있다.


In [ ]:
# 4. retrieve relevant documents
query = "How does Netflix use Java?"
results = vectorstore.similarity_search(query)

results

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[Document(id='0fbe2dce-caee-4350-876c-dd1d63767a9a', metadata={'category': 'Title', 'file_directory': './files', 'filename': 'How_Neflix_Uses_Java_-_2026_.pdf', 'filetype': 'application/pdf', 'last_modified': '2026-05-15T14:39:09', 'element_id': '0fb1670ce38e727961056ab17bc4bc8c', 'source': './files/How_Neflix_Uses_Java_-_2026_.pdf', 'page_number': 2}, page_content='How Neflix Uses Java - 2026 Edition'),
 Document(id='9c9315a9-e843-4f86-bbf7-dc4af15ad43b', metadata={'file_directory': './files', 'filetype': 'application/pdf', 'filename': 'How_Neflix_Uses_Java_-_2026_.pdf', 'source': './files/How_Neflix_Uses_Java_-_2026_.pdf', 'last_modified': '2026-05-15T14:39:09', 'category': 'Title', 'page_number': 3, 'element_id': '0fb1670ce38e727961056ab17bc4bc8c'}, page_content='How Neflix Uses Java - 2026 Edition'),
 Document(id='b6f86a8d-b38d-4f72-bdd9-859385df9fc8', metadata={'category': 'Title', 'element_id': '0fb1670ce38e727961056ab17bc4bc8c', 'page_number': 4, 'source': './files/How_Neflix_Us